# variableMeanDriftingGrating

Protocol-specific analysis. `demos/meaAnalysisMain.ipynb` is the shared front half — it finds datasets, builds a pipeline and checks that the noise chunk and the protocol datafile describe the same cells. That notebook deliberately stops before interpreting conditions, because that is where protocols stop resembling each other. This is where it picks up.

The protocol drifts a sinewave grating at a fixed temporal frequency while alternating two things across epochs: the **background mean intensity** and the **bar width**. So the design is mean × bar width, and every epoch belongs to one cell of that grid.

**This notebook loads its own data.** Copy the experiment and datafile out of the main notebook's §6 printout into the constants below and run from the top — nothing is inherited from another kernel. If you *are* in the same kernel and already have a `pipeline`, the load cell will reuse it rather than rebuild.

The order here is deliberate: read the protocol source, get the condition axes from it, then clean — **epochs first, then cells**. Cleaning the other way round scores every cell against a stretch of block you were going to discard anyway, which reports a property of the block as a property of each cell.

## 1. Setup and the dataset

`PROTOCOL_NAME` is the full dotted name as the database stores it — §2 uses it to find the MATLAB source, so it has to be the real one, not the search fragment.

`create_mea_pipeline` is the same call the main notebook's §6 makes. Pinning `ANALYSIS_CHUNK` is optional; leave it `None` and the nearest noise chunk with a typing file is chosen, by the same rule the main notebook uses.

In [10]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# <-- EDIT ME: from the main notebook's §6 printout.
EXP_NAME       = '20230502C'
DATAFILE_NAME  = 'data017'
ANALYSIS_CHUNK = 'chunk2'     # None to let the pipeline pick

PROTOCOL_NAME = 'edu.washington.riekelab.chris.protocols.variableMeanDriftingGrating'
MAIN_TYPES    = ['OnP', 'OffP', 'OnM', 'OffM']

# Reuse a pipeline from the same kernel when it is already the right dataset;
# otherwise build one. Makes the notebook runnable standalone without
# rebuilding needlessly when it isn't.
_existing = globals().get('pipeline')
if (_existing is not None
        and getattr(_existing.resp, 'datafile_name', None) == DATAFILE_NAME
        and _existing.analysis_chunk.exp_name == EXP_NAME):
    print(f'Reusing the pipeline already in this kernel: {EXP_NAME}/{DATAFILE_NAME}')
else:
    pipeline = ra.create_mea_pipeline(EXP_NAME, DATAFILE_NAME,
                                      analysis_chunk_name = ANALYSIS_CHUNK)

stim_block     = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk

Reusing the pipeline already in this kernel: 20230502C/data017


## 2. What the block ran

Everything here comes from the recorded epochs. The MATLAB source declares defaults, but a default is only what a parameter would have been had nobody touched it, and on a real rig most of them are touched — this protocol declares a 4 Hz temporal frequency and an 800 µm aperture; the block below ran 2 Hz and 2000. Reading defaults back as if they described the experiment is how you report a stimulus the retina never saw.

The source is still worth opening for the one thing the data cannot supply: the comment beside each property, which says what the parameter *is*. `ra.parse_protocol_source` finds the `.m` in the locally cloned package — the dotted protocol name maps straight onto MATLAB's package layout, so `edu.washington.riekelab.chris.protocols.X` is `+edu/+washington/+riekelab/+chris/+protocols/X.m` under `chris-package` — and prints the GitHub URL so the source of truth is one click away: <https://github.com/Rieke-Lab/chris-package/>. Skip it and you get the same tables minus the descriptions.

**Two tables, because a block has two kinds of parameter.**

The first is the **settings**: the protocol's own parameters that held still for the whole block. Only the ones this protocol declares — a block records far more than the protocol sets, and rig geometry, display calibration and Symphony's bookkeeping would bury ten protocol parameters in thirty rows of context. Pass `declared_only=False` for the full record, which is where `NDF`, `micronsPerPixel` and `monitorRefreshRate` live. Parameters the protocol *inherits* rather than declares — `preTime` and `tailTime` come from `RiekeLabStageProtocol` — count as undeclared, since only this protocol's own `.m` is read; they are still in `stim_block.d_epoch_block_params`, which is where §4 reads the epoch length from.

The second is **per epoch**: one row each, carrying the values of the parameters that *changed* — those are the condition axes, and a parameter qualifies by varying in the data rather than by being declared to vary — alongside the population spike count for that epoch. The two halves of an epoch belong together: what was shown, and what came back. Keeping them in one table is what makes §3's epoch range choosable by eye instead of by cross-referencing a table against a plot.

Worth knowing for this protocol: it writes `epoch.addParameter('currentBarWdith', ...)`, misspelled in the MATLAB, so `currentBarWdith` is the column name and anything reaching for `currentBarWidth` silently gets nothing. `condition_keys` reports any name the source declares per-epoch that does not vary in the data, which is how a mismatch like that surfaces.

In [11]:
source = ra.parse_protocol_source(PROTOCOL_NAME)

if source is not None:
    print(f'{source.class_name}  <  {source.superclass}')
    print(f'github : {source.github_url}\n')

params = ra.block_parameters(stim_block, source = source)

# Fixed for the whole block — the protocol's own configuration.
print(f"{len(params)} parameters declared by {source.class_name if source else 'the protocol'}; "
      f"{params.attrs.get('n_undeclared', 0)} more the rig recorded are left out "
      f"(declared_only=False to see them).\n")
print(f'Settings held constant across all {len(stim_block.df_epochs)} epochs:')
display(params.query('not epoch_specific')[['parameter', 'value', 'comment']]
              .reset_index(drop = True))

# Varying — the condition axes, with the levels each one took.
print('\nParameters that change from epoch to epoch (the condition axes):')
display(params.query('epoch_specific')[['parameter', 'value', 'n_levels', 'comment']]
              .reset_index(drop = True))

CONDITION_KEYS = ra.condition_keys(stim_block, source = source)

variableMeanDriftingGrating  <  edu.washington.riekelab.protocols.RiekeLabStageProtocol
github : https://github.com/Rieke-Lab/chris-package/blob/master/+edu/+washington/+riekelab/+chris/+protocols/variableMeanDriftingGrating.m

13 parameters declared by variableMeanDriftingGrating; 15 more the rig recorded are left out (declared_only=False to see them).

Settings held constant across all 20 epochs:


,parameter,value,comment
0,amp,Amp1,Output amplifier
1,apertureDiameter,2000.0,Surround radius (pix)
2,barWidths,"[50.0, 150.0]",Center bar width (pix)
3,meanIntensities,"[0.03, 0.3]",Background light intensity (0-1)
4,numberOfEpochs,20,Number of epochs
5,onlineAnalysis,extracellular,Online analysis type.
6,orientation,0.0,Center orientation (deg)
7,spatialClass,sinewave,Grating spatial type
8,spatialContrast,0.9,Center grating contrast (0-1)
9,stimTime,60000.0,Stimulus duration (ms)



Parameters that change from epoch to epoch (the condition axes):


,parameter,value,n_levels,comment
0,currentBarWdith,"[50.0, 150.0]",2,
1,currentMeanIntensity,"[0.03, 0.3]",2,


In [12]:
# One row per epoch: the conditions it ran, and what the population did.
epochs = ra.epoch_condition_table(stim_block, response_block,
                                  cell_types = MAIN_TYPES, minimum_n = 3,
                                  source = source)
display(epochs)

# The design as it ran. An unbalanced grid here is worth noticing before it
# becomes an unbalanced comparison later.
conditions = {key: epochs[key].to_numpy() for key in CONDITION_KEYS}
print(f'{len(epochs)} epochs over the condition grid:')
display(pd.crosstab(epochs[CONDITION_KEYS[0]], epochs[CONDITION_KEYS[1]]))

,epoch,currentBarWdith,currentMeanIntensity,n_spikes,spikes_per_cell
0,0,50.0,0.03,14707,56.3
1,1,50.0,0.30,32183,123.3
2,2,150.0,0.03,1200,4.6
3,3,150.0,0.30,64461,247.0
4,4,50.0,0.03,4319,16.5
5,5,50.0,0.30,50411,193.1
6,6,150.0,0.03,13530,51.8
7,7,150.0,0.30,114439,438.5
8,8,50.0,0.03,20176,77.3
9,9,50.0,0.30,80745,309.4


20 epochs over the condition grid:


currentMeanIntensity,0.03,0.30
currentBarWdith,,
50.0,5,5
150.0,5,5


## 3. Choose the epochs to analyze

The `n_spikes` column in §2 is the whole input to this decision, so this section is just the decision. Set `EPOCH_RANGE` to the stretch of the block you want; everything downstream is scored on it.

In [15]:
# <-- EDIT ME: the epochs to analyze, read off the n_spikes column in §2.
EPOCH_RANGE = (7, 20)

kept = slice(*EPOCH_RANGE)
epochs_kept = epochs.iloc[kept]
print(f'analyzing epochs {EPOCH_RANGE[0]}\u2013{EPOCH_RANGE[1] - 1} '
      f'({len(epochs_kept)} of {len(epochs)})\n')

# What the trim left in each cell of the condition grid, and how hard the
# population fired there. Uneven counts here unbalance any comparison across
# conditions made downstream.
display(epochs_kept.groupby(CONDITION_KEYS)['n_spikes']
                   .agg(n_epochs = 'size', median_spikes = 'median')
                   .astype({'median_spikes': int}))

analyzing epochs 7–19 (13 of 20)



n_epochs  median_spikes
currentBarWdith currentMeanIntensity                         
50.0            0.03                         3          32153
                0.30                         3          90061
150.0           0.03                         3          34122
                0.30                         4         143558

## 4. Drop the silent cells

**One criterion, so the decision is inspectable.** A cell counts as *active* in an epoch when it fires above `MIN_RATE_HZ`, and is kept when it is active in at least `MIN_ACTIVE_FRACTION` of the analyzed epochs. Every other gate `QCThresholds` offers — burstiness, drift, longest silent run — is switched off here. They are defensible checks, but stacking six of them makes a rejection impossible to attribute, and the thing being removed is cells that barely fire.

The rate threshold is a *rate*, not a count, so it scales with epoch length: the per-epoch bar is `MIN_RATE_HZ × 60 s` = 60 spikes on this protocol, and the same setting transfers to a protocol with 2 s epochs without retuning.

**Why the fraction is not 0.8.** The conditions alternate epoch to epoch, and one of them is dim. Demanding activity in 80% of epochs demands it in the dim epochs too, which is a response requirement wearing a quality gate's clothes — at 0.8 this dataset keeps 20 of 261 cells and throws away ones with a median rate of 8 Hz. `0.8 / n_levels` asks for the same 80% within one condition's worth of epochs, so a cell that responds properly to the bright condition and goes quiet in the dim one is kept. The dominant axis is measured rather than assumed, because that level count is what sizes the gate.

**Then look at what it did.** The dropdown renders the analyzed epochs for either group of any cell type — pick a `dropped` entry and the rows should be visibly empty or near it. If dropped cells are firing perfectly well, the threshold is wrong, not the cells. Rows are cell IDs and the panels are the epochs `EPOCH_RANGE` selected, so the picture is of the same data the gate scored.

Read the summary table before moving on. If a cell type lost most of its cells, any population claim about that type from here rests on whatever is left, and the honest move is to say so — or to loosen the gate and say that instead.

In [ ]:
# Which condition axis actually drives firing rate? The gate needs to know how
# many levels a healthy cell could reasonably be silent in, and that is a
# property of the strongest axis. Measured, not assumed — picking whichever
# axis sorted first would have chosen bar width here.
pop = epochs['n_spikes'].to_numpy()
spread = {}
for key in CONDITION_KEYS:
    medians = [np.median(pop[epochs[key].to_numpy() == level])
               for level in sorted(set(conditions[key]))]
    spread[key] = max(medians) / min(medians) if min(medians) > 0 else np.inf
DOMINANT_AXIS = max(spread, key = spread.get)
for key, ratio in sorted(spread.items(), key = lambda kv: -kv[1]):
    print(f'{key:24s} changes population rate {ratio:5.1f}x across its levels')

n_levels = len(set(conditions[DOMINANT_AXIS][slice(*EPOCH_RANGE)]))

# <-- EDIT ME: the whole criterion.
MIN_RATE_HZ         = 1.0                      # a cell is "active" above this
MIN_ACTIVE_FRACTION = round(0.8 / n_levels, 2) # ... in this fraction of epochs

# Epoch length from the protocol's own timing. Pass this explicitly: the rate
# gate divides by it, and without it every rate metric is NaN and the whole
# block fails QC silently.
block = stim_block.d_epoch_block_params
T_END_MS = sum(float(block.get(k, 0) or 0) for k in ('preTime', 'stimTime', 'tailTime'))

print(f'-> {DOMINANT_AXIS} has {n_levels} levels, so a cell must be active in '
      f'{MIN_ACTIVE_FRACTION:.0%} of epochs')
print(f'   active = more than {MIN_RATE_HZ:g} Hz over a '
      f'{T_END_MS / 1000:.0f} s epoch ({MIN_RATE_HZ * T_END_MS / 1000:.0f} spikes)\n')

# One gate on, the rest off, so a rejection has exactly one cause.
thresholds = ra.QCThresholds(
    min_rate_hz                = MIN_RATE_HZ,
    min_frac_epochs_above_rate = MIN_ACTIVE_FRACTION,
    min_frac_non_silent_epochs = None,
    max_cv                     = None,
    max_silent_trial_frac      = None,
    max_silent_run             = None,
    max_drift_score            = None,
    min_reliability_r          = None,
)

qc = ra.block_qc_metrics(response_block, cell_types = MAIN_TYPES,
                         epoch_range = EPOCH_RANGE, t_end_ms = T_END_MS,
                         min_rate_hz = MIN_RATE_HZ)
qc = ra.filter_cells_by_qc(qc, thresholds)

GOOD_CELLS = qc.query('passes')['cell_id'].astype(int).tolist()
print(f'{len(GOOD_CELLS)} of {len(qc)} cells kept\n')

display(qc.assign(group = np.where(qc['passes'], 'kept', 'dropped'))
          .groupby(['cell_type', 'group'])
          .agg(n = ('cell_id', 'size'),
               median_rate_hz = ('mean_rate_hz', 'median'),
               median_active_frac = ('frac_epochs_above_rate', 'median'))
          .round(2))

In [ ]:
# Look at both groups. A 'dropped' panel should be visibly empty; if those
# cells are firing well, the threshold is wrong rather than the cells.
groups = {}
for cell_type in sorted(qc['cell_type'].dropna().unique()):
    rows = qc[qc['cell_type'] == cell_type]
    for label in ('kept', 'dropped'):
        ids = rows[rows['passes'] == (label == 'kept')]['cell_id'].astype(int).tolist()
        if ids:
            groups[f'{cell_type} \u2014 {label} ({len(ids)} cells)'] = (cell_type, ids)


def _render(key):
    cell_type, ids = groups[key]
    fig = ra.plot_epoch_rasters(response_block, cell_type, cell_ids = ids,
                                epoch_range = EPOCH_RANGE,
                                n_first = 3, n_last = 3,
                                title = f'{key}, epochs '
                                        f'{EPOCH_RANGE[0]}\u2013{EPOCH_RANGE[1] - 1}')
    return None, ra.figure_to_png(fig)


ra.png_browser([(label, label) for label in groups], _render,
               description = 'Show:');

## 5. Where to go from here

`EPOCH_RANGE`, `GOOD_CELLS` and `conditions` are the cleaned handles. Everything downstream should be written against those three rather than against the raw block, so the cleaning applies once and visibly.

The obvious next step for this protocol is the response as a function of the two condition axes — PSTHs per (mean intensity × bar width) cell of the grid, and a summary of how mean intensity shifts the grating response at each bar width. `ra.get_spike_xarr(response_block, cell_types=MAIN_TYPES)` gives a ragged (cell × epoch) array to build that on; slice it with `EPOCH_RANGE` and select cells with `GOOD_CELLS`.

Worth carrying forward, from what the cleaning showed: the tenfold rise in population rate across this block is monotonic and in both conditions. Any comparison between conditions is safe — they alternate, so both are sampled evenly across the trend — but a comparison between *early and late* epochs is confounded with it.